# C3 · Clip, mask and subset: cutting the DEM to the Kathmandu Valley and its districts

> **Time:** about 2 hours  
> **You need:** Lessons C1 and C2 (windows build on C1's transform; rioxarray was introduced in C2), B5 · Selecting by attribute and B7 · Clip and select by location.  
> **By the end you can:**
> 1. read only a part of a raster with a **window**, and crop a raster to a box with rioxarray's `clip_box`,
> 2. **clip** a raster to district polygons, decide which edge cells count (`all_touched`), and keep the cells outside as `NaN` rather than as nodata,
> 3. summarise districts that lie only partly inside a raster honestly, and save a clipped raster with and without compression.

In B7 you clipped vector layers: landslide points to a district, districts to a box. Rasters are clipped too, for three reasons. A smaller raster is faster to read and process. Statistics for a district must use only that district's cells. And a map of one area should not carry 3.5 million cells you do not need.

This lesson cuts the DEM in four ways: by rows and columns, by a box, by a polygon, and by a polygon that reaches beyond the edge of the raster.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).* The full licence notice is in `data/README.md`.

**Counting lines:** as in C2, plot setup lines (`fig, ax = plt.subplots()`, `ax.set_aspect(aspect)`, `plt.show()`) do not count toward the Q1/Q2/Q3 line counts. Neither does a `with rasterio.open(...) as src:` line.

**How to practise:** after every step there are three questions. **Q1** needs one line of code, **Q2** two lines, **Q3** about three. Type your answer in the empty cell under each question, run it, then open **Show solution** to check. Try first, even if you are not sure: struggling for a few minutes is how the skill grows.

<!-- pygile-nav -->
**PyGILE Course** · lesson 24 of 29 · [Course home](../README.md)  
← Previous: [C2 · Plotting rasters: an honest map of the Kathmandu Valley DEM](C02_Plotting_rasters.ipynb) · Next: [C4 · Reproject, resample and mosaic: the DEM on a grid in metres](C04_Reproject_resample_and_mosaic.ipynb) →

## 1. Reading only a window

A **window** is a rectangle of cells inside a raster, given by where it starts and how big it is: `Window(col_off, row_off, width, height)`. `col_off` and `row_off` are the **offsets**: the column and row of its top-left cell.

You rarely know the rows and columns of the area you want, but you know its coordinates. `rasterio.windows.from_bounds(left, bottom, right, top, transform=src.transform)` turns a box in map coordinates into a window.

A box edge can fall inside a cell, so the window may start at a fraction such as row 648.5. Round it to whole cells with `.round_offsets().round_lengths()`. Reading a fractional window makes rasterio **resample** the values, and they may shift by a cell.

`src.read(1, window=win)` then reads **only those cells** from the file. For a satellite mosaic of many gigabytes, that is the only way to work on a laptop. `src.window_transform(win)` gives the transform of the piece: same cell size, new top-left corner.

Below, the window is the Kathmandu Valley box from B2 (85.2–85.55° E, 27.55–27.82° N, approximate).

In [ ]:
import math
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import rasterio
import rioxarray as rxr
from matplotlib.colors import ListedColormap
from rasterio.plot import plotting_extent
from rasterio.transform import rowcol
from rasterio.windows import Window, from_bounds

DATA = Path("..") / "data"
OUT = Path("..") / "outputs"
OUT.mkdir(exist_ok=True)
DEM_PATH = DATA / "dem_kathmandu_melamchi.tif"

with rasterio.open(DEM_PATH) as src:                  # the whole DEM, as in C1
    dem_m = src.read(1)
    dem_transform = src.transform
    bounds = src.bounds

aspect = 1 / math.cos(math.radians((bounds.top + bounds.bottom) / 2))          # C2, step 3
land_cmap = ListedColormap(plt.colormaps["terrain"](np.linspace(0.25, 1, 256)))   # C2, step 4
districts = gpd.read_file(DATA / "nepal_districts.geojson")

In [ ]:
with rasterio.open(DEM_PATH) as src:
    win = from_bounds(85.2, 27.55, 85.55, 27.82, transform=src.transform)
    print("as computed:", round(win.col_off, 2), round(win.row_off, 2), round(win.width, 2), round(win.height, 2))
    win = win.round_offsets().round_lengths()
    print("rounded:", win)
    valley_m = src.read(1, window=win)
    valley_transform = src.window_transform(win)

print(valley_m.shape, valley_m.min(), valley_m.max())

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(valley_m, extent=plotting_extent(valley_m, valley_transform), cmap=land_cmap)
fig.colorbar(im, ax=ax, label="Elevation (m)")
ax.set_aspect(aspect)
ax.set_title("Window read: Kathmandu Valley box (approximate)")
plt.show()

The window first came out as column 0.5, row 648.5, 1,260 columns wide and 972 rows high. The box edges fall halfway across cells, because the DEM's cell **centres** sit on whole seconds of arc (C1). Rounding gave whole cells: column 0, row 648.

Only 972 × 1,260 cells were read, from 850 m to 2,762 m. The map is the same as Review 1 of C2, where you cut the slice yourself with `rowcol`. `plotting_extent` (C2) built the extent from the window's own transform.

*Elevation: Copernicus DEM GLO-30 (© DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the European Union and ESA).*

### Practice

**Q1 · one line:** How much memory does `valley_m` take, compared with the whole DEM? Print `valley_m.nbytes / dem_m.nbytes`, rounded to 2 decimal places.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(valley_m.nbytes / dem_m.nbytes, 2))
```

0.35: the window holds 35% of the cells, so it needs 35% of the memory (about 2.4 MB instead of 7.0 MB).
</details>

**Q2 · two lines:** Compare the top-left corners: print `valley_transform.c` and `.f`, then `dem_transform.c` and `.f`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(valley_transform.c, valley_transform.f)
print(dem_transform.c, dem_transform.f)
```

Both start at the same longitude, 85.1998…°, because the window starts at column 0. The window's top is at 27.8201…° instead of 28.0001…°: 648 rows × 1/3600° lower. Cell sizes `a` and `e` stay the same.
</details>

**Q3 · three lines:** Find Kathmandu Durbar Square (85.307° E, 27.704° N) in the **window**: use `rowcol` with `valley_transform`, then print the row, the column and the elevation from `valley_m`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
row, col = rowcol(valley_transform, 85.307, 27.704)
print(row, col, valley_m[row, col])
```

Row 418, column 385, elevation 1,300 m. In C1 the same cell was row 1066 of the whole DEM: 1066 − 648 = 418. Each piece needs its own transform, or every position is off.
</details>

## 2. Windows and slices are the same idea

A window is a NumPy slice (A8) that rasterio applies while reading the file:

`Window(col_off, row_off, width, height)` ↔ `array[row_off:row_off + height, col_off:col_off + width]`

Watch the order: `Window` takes the **column first** (x first, like `transform @ (col, row)` in C1), but a NumPy slice takes the **row first**. `Window.from_slices((row_start, row_stop), (col_start, col_stop))` builds a window from row and column ranges, rows first, like a slice.

The difference is memory. A slice needs the whole array in memory first. A window reads only its cells from the disk.

**Predict:** the window below starts at column 1,860, row 0, and is 300 cells square. Which corner of the DEM is that?

In [ ]:
print(np.array_equal(valley_m, dem_m[648:648 + 972, 0:0 + 1260]))

corner_win = Window(col_off=1860, row_off=0, width=300, height=300)
with rasterio.open(DEM_PATH) as src:
    corner_m = src.read(1, window=corner_win)
    print(corner_m.shape, corner_m.max(), round(corner_m.mean(), 1))
    print(src.window_bounds(corner_win))

The window read and the slice are exactly equal (`True`).

Column 1,860 plus 300 columns reaches column 2,159, the last one, and row 0 is the top: the **north-east corner**. `window_bounds` confirms it: 85.72° to 85.80° E, 27.92° to 28.00° N. It holds the highest cell of the DEM, 3,979 m (C1), and its mean is 2,599.1 m.

### Practice

**Q1 · one line:** Write the NumPy slice that matches `corner_win`, and check with `np.array_equal` that it gives the same values as `corner_m`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
np.array_equal(corner_m, dem_m[0:300, 1860:2160])
```

`True`. Rows 0 to 299, then columns 1,860 to 2,159: in the slice the rows come first.
</details>

**Q2 · two lines:** Build a window with `Window.from_slices((1000, 1100), (300, 500))` and print its `width` and `height`.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
win_2 = Window.from_slices((1000, 1100), (300, 500))
print(win_2.width, win_2.height)
```

200 and 100: the rows (1000 to 1100) give the height, the columns (300 to 500) the width. Printing `win_2` shows `Window(col_off=300, row_off=1000, width=200, height=100)`, columns first again.
</details>

**Q3 · three lines:** Read a 100 × 100 window around Durbar Square, whose cell is row 1066, column 385 (C1): rows 1016 to 1115, columns 335 to 434. Print its shape, lowest, highest and mean elevation, rounded to 1 decimal place.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
with rasterio.open(DEM_PATH) as src:
    durbar_m = src.read(1, window=Window.from_slices((1016, 1116), (335, 435)))
print(durbar_m.shape, durbar_m.min(), durbar_m.max(), round(durbar_m.mean(), 1))
```

(100, 100), from 1,272 to 1,325 m, mean 1,298.0 m. About 3 km × 3 km of the valley floor, with only 53 m between the lowest and highest cell: the flattest ground in the DEM.
</details>

## 3. Cropping a DataArray to a box: `clip_box`

In C2 you opened the DEM with rioxarray as a **DataArray**. Its cut-out tools work with coordinates, and every piece keeps its own coordinates, CRS and transform. No bookkeeping for you.

`dem_da.rio.clip_box(minx=..., miny=..., maxx=..., maxy=...)` crops to a box. The four numbers come in the order of `bounds`: left, bottom, right, top.

You can also select by coordinate labels with `.sel` and `slice` (as with `.loc` in A9): `dem_da.sel(x=slice(85.2, 85.55), y=slice(27.82, 27.55))`. The y slice runs from **top to bottom**, 27.82 then 27.55, because the y coordinates decrease down the rows (C2, step 7).

In [ ]:
dem_da = rxr.open_rasterio(DEM_PATH, masked=True).squeeze("band", drop=True)   # as in C2

valley_da = dem_da.rio.clip_box(minx=85.2, miny=27.55, maxx=85.55, maxy=27.82)
print(valley_da.shape)
print(valley_da.rio.bounds())
print(float(valley_da.min()), float(valley_da.max()), round(float(valley_da.mean()), 1))

In [ ]:
by_label = dem_da.sel(x=slice(85.2, 85.55), y=slice(27.82, 27.55))
print(by_label.shape)

wrong_way = dem_da.sel(x=slice(85.2, 85.55), y=slice(27.55, 27.82))   # y in the wrong order, ON PURPOSE
print(wrong_way.shape)

`clip_box` gave 972 rows and **1,261** columns, one column more than the window of step 1. That extra column even holds a lower cell: the minimum is 849 m here, against 850 m in the window. The box's east edge, 85.55°, falls exactly on the centres of one column of cells: `clip_box` keeps that column, and the rounded window stops one column short of it. Edge cells often cause such off-by-one differences, so always check shapes.

`.sel` with the y slice from top to bottom gave the same 972 × 1,261 cells. With the y slice the wrong way round, it gave **0 rows**, and no error. An empty result is the sign that a slice runs against the order of the coordinates.

### Practice

**Q1 · one line:** Crop `dem_da` to a small box around Durbar Square, from 85.297 to 85.317° E and from 27.694 to 27.714° N, and show its shape.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
dem_da.rio.clip_box(minx=85.297, miny=27.694, maxx=85.317, maxy=27.714).shape
```

(73, 73): 0.02° is 72 cells of 1/3600°, plus one for the cells cut by the edges. On the ground that is about 2.0 km × 2.2 km.
</details>

**Q2 · two lines:** Crop `dem_da` to the high north-east, 85.6–85.8° E and 27.85–28.0° N. Print the highest and the mean elevation, the mean rounded to 1 decimal place.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
ne_da = dem_da.rio.clip_box(minx=85.6, miny=27.85, maxx=85.8, maxy=28.0)
print(float(ne_da.max()), round(float(ne_da.mean()), 1))
```

3,979 m and 2,138.8 m. `float(...)` turns the one-value DataArray that `.max()` returns into a plain number. The mean is about 600 m above the mean of the whole DEM (1,531.2 m, C1).
</details>

**Q3 · three lines:** Draw `valley_da` with `.plot.imshow` (C2), in `land_cmap` colours, with a colourbar label and true proportions.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
fig, ax = plt.subplots()
valley_da.plot.imshow(ax=ax, cmap=land_cmap, cbar_kwargs={"label": "Elevation (m)"})
ax.set_aspect(aspect)
plt.show()
```

No extent to work out: the DataArray carries its own x and y coordinates, so the axes show 85.2 to 85.55 and 27.55 to 27.82 by themselves.
</details>

## 4. Clipping to a district polygon

A box is rarely the shape you need. `dem_da.rio.clip(shapes, crs=...)` **clips** a raster to polygons: cells inside keep their values, cells outside become `NaN`.

- `shapes` is a GeoSeries of polygons, such as one district's `.geometry` (B5 selection).
- `crs=` says which CRS the polygons use. Bring them to the raster's CRS first with `to_crs` (B4) if needed; here both are EPSG:4326.
- A cell counts as inside when its **centre** lies inside the polygon.
- By default the result is also **cropped** to the polygon's extent (`drop=True`). With `drop=False` it keeps the full grid, and everything outside the polygon is `NaN`.

The word **mask** is used for this too: the polygon works like a stencil, hiding the cells outside.

In [ ]:
kathmandu = districts[districts["district"] == "Kathmandu"]
print(kathmandu.crs == dem_da.rio.crs)

ktm_da = dem_da.rio.clip(kathmandu.geometry, crs=kathmandu.crs)
print(ktm_da.shape)
print("cells inside:", int(ktm_da.notnull().sum()))
print(float(ktm_da.min()), float(ktm_da.max()), round(float(ktm_da.mean()), 1))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
ktm_da.plot.imshow(ax=ax, cmap=land_cmap, cbar_kwargs={"label": "Elevation (m)"})
kathmandu.boundary.plot(ax=ax, color="black", linewidth=1)
ax.set_aspect(aspect)
ax.set_title("DEM clipped to Kathmandu district")
plt.show()

In [ ]:
ktm_full = dem_da.rio.clip(kathmandu.geometry, crs=kathmandu.crs, drop=False)
print(ktm_full.shape, int(ktm_full.notnull().sum()))

The clipped grid has 888 rows and 1,317 columns, the district's extent. Of these cells, 488,217 lie inside the district. They run from 842 m to 2,726 m, with a mean of 1,553.4 m.

On the map, the cells outside the outline are blank: `NaN` cells get no colour (C2, Review 3). The district's thin western tip, beyond 85.2° E, is blank too: it lies outside the DEM (step 6). `.notnull()` is `True` for every cell with a value, so its sum counts the cells inside.

With `drop=False` the grid keeps all 1,620 × 2,160 cells, but the count of cells with a value is the same: 488,217. Crop unless you need the clip to line up with the full DEM.

### Practice

**Q1 · one line:** How many cells of `ktm_da` are `NaN`? Use `.isnull()` and `.sum()`, inside `int()`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
int(ktm_da.isnull().sum())
```

681,279 cells: the corners of the district's box that lie outside the district. Inside plus outside, 488,217 + 681,279, gives all 888 × 1,317 = 1,169,496 cells.
</details>

**Q2 · two lines:** Clip the DEM to **Bhaktapur** district and print its highest and mean elevation, the mean rounded to 1 decimal place.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
bkt_da = dem_da.rio.clip(districts[districts["district"] == "Bhaktapur"].geometry, crs=districts.crs)
print(float(bkt_da.max()), round(float(bkt_da.mean()), 1))
```

2,147 m and 1,445.5 m: a lower and flatter district than Kathmandu, mostly on the valley floor. The selection from B5 can go straight inside `clip`.
</details>

**Q3 · three lines:** What share of Kathmandu district lies above 2,000 m, and what share below 1,400 m? Divide each count by the number of cells inside, and print both as percentages rounded to 1 decimal place.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
n_inside = int(ktm_da.notnull().sum())
print(round(int((ktm_da > 2000).sum()) / n_inside * 100, 1))
print(round(int((ktm_da < 1400).sum()) / n_inside * 100, 1))
```

11.1% above 2,000 m and 47.1% below 1,400 m. Comparing with `NaN` always gives `False`, so the cells outside the district are never counted. Divide by the cells **inside**, not by the size of the grid.
</details>

## 5. Outside cells: `NaN` or nodata?

What a clip writes into the cells outside depends on how the raster was opened.

- With `masked=True` (C2), they become `NaN`, and the values are decimals (`float32`). `mean`, `min` and `max` skip `NaN`.
- Without it, rioxarray keeps the whole numbers (`int16`) and fills the outside with the **nodata** value, −32768. Every calculation then counts −32768 as an elevation, the trap you met in C1, step 7.

Rasterio has its own clipping tool, `rasterio.mask.mask(src, shapes, crop=True)`. It returns two things: the array, with outside cells set to nodata, and the transform of the cropped piece. The array is 3-D, (bands, rows, columns), like `src.read()`. With `filled=False` it returns a NumPy **masked array** instead, which hides the outside cells from calculations.

**Predict:** what will the mean of the clip be without `masked=True`?

In [ ]:
raw_da = rxr.open_rasterio(DEM_PATH).squeeze("band", drop=True)    # NO masked=True
ktm_raw = raw_da.rio.clip(kathmandu.geometry, crs=kathmandu.crs)
print(ktm_raw.dtype, ktm_raw.rio.nodata)
print("mean, nodata counted as elevation:", round(float(ktm_raw.mean()), 1))

raw_values = ktm_raw.values                     # the NumPy array inside the DataArray
print("mean of valid cells only:", round(raw_values[raw_values != -32768].mean(), 1))

In [ ]:
from rasterio.mask import mask

with rasterio.open(DEM_PATH) as src:
    ktm_arr, ktm_transform = mask(src, kathmandu.geometry, crop=True)

print(type(ktm_arr), ktm_arr.shape, ktm_arr.dtype)
print("nodata cells:", (ktm_arr == -32768).sum())

Without `masked=True`, the clip stays `int16` with nodata −32768, and its mean is **−18,440.2 m**: impossible, yet no error warns you. Leaving out the nodata cells gives back 1,553.4 m. `.values` gives the plain NumPy array inside a DataArray.

`mask` returned a 3-D `int16` array of shape (1, 890, 1318), with 684,803 nodata cells around the district. It is two rows and one column bigger than rioxarray's clip: rasterio crops to the polygon's bounding box rounded **outwards** to whole cells, while rioxarray trims to the rows and columns that keep at least one cell. The valid cells are the same.

The safest habit: open with `masked=True`, and check `min()` after every clip. A minimum of −32768 means nodata got into your numbers.

### Practice

**Q1 · one line:** Print the lowest value of `ktm_raw`, as an `int`.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(int(ktm_raw.min()))
```

−32768, the nodata value. A minimum far below any real elevation is the quickest warning sign.
</details>

**Q2 · two lines:** Find the mean of the valid cells of `ktm_arr` (from `mask`) with a True/False mask, as in C1. Round to 1 decimal place.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
valid = ktm_arr != -32768
print(round(ktm_arr[valid].mean(), 1))
```

1,553.4 m, the same as with rioxarray and `masked=True`. Both tools keep the same cells; only the packaging differs.
</details>

**Q3 · three lines:** Call `mask` again with `filled=False` to get a masked array. Print its mean rounded to 1 decimal place, and the number of valid cells with `.count()`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
with rasterio.open(DEM_PATH) as src:
    ktm_ma, _ = mask(src, kathmandu.geometry, crop=True, filled=False)
print(round(float(ktm_ma.mean()), 1))
print(ktm_ma.count())
```

1,553.4 m from 488,217 valid cells, the same as `ktm_da`. A masked array remembers which cells to skip. `_` is a name for a result you do not need, here the transform.
</details>

## 6. Which edge cells count? `all_touched`

A district border cuts through thousands of cells. Which of them belong to the district?

- **Default:** a cell is inside when its **centre** is inside the polygon. Cells cut by the border are kept about as often as they are dropped, so areas and averages come out fair.
- **`all_touched=True`:** every cell the polygon touches is kept, even a corner. Nothing inside is missed, but the clip is a little too big.

The difference is a thin ring of cells along the border. It matters more for small polygons and for large cells. For statistics, keep the default. Use `all_touched=True` when no cell may be missed, for example when checking whether any steep ground touches a site.

Cell counts become areas with the cell size from C1: about 27.3 m × 30.8 m, so 0.00084 km² per cell (approximate). In B4 you measured Kathmandu district at about 414.3 km² in UTM 45N.

In [ ]:
ktm_all = dem_da.rio.clip(kathmandu.geometry, crs=kathmandu.crs, all_touched=True)

n_centre = int(ktm_da.notnull().sum())
n_touched = int(ktm_all.notnull().sum())
cell_km2 = 27.3 * 30.8 / 1000000                  # approximate cell area, from C1
print(n_centre, n_touched, n_touched - n_centre)
print(round(n_centre * cell_km2, 1), "km2 (centres)", round(n_touched * cell_km2, 1), "km2 (all touched)")

kathmandu_km2 = kathmandu.to_crs(32645).area.iloc[0] / 1000000   # as in B4
print(round(kathmandu_km2, 1), "km2 from the polygon in UTM 45N")

bhaktapur = districts[districts["district"] == "Bhaktapur"]       # for the practice below

`all_touched=True` adds 2,662 cells along the border, about 0.5% more.

The cells give 410.5 km² (centres) and 412.8 km² (all touched), against 414.3 km² for the polygon. Two things explain the gap. A thin sliver of Kathmandu district lies west of the DEM's edge, so its cells are missing (you saw it in C2). And the cell area is an approximation: cells shrink a little towards the north (C1).

So a raster can measure areas, but check the result against the vector polygon once.

### Practice

**Q1 · one line:** Using `n_centre` and `n_touched`, print by how many percent `all_touched` enlarges the Kathmandu clip, rounded to 2 decimal places.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round((n_touched - n_centre) / n_centre * 100, 2))
```

0.55%. For a district of half a million cells, the border ring is small.
</details>

**Q2 · two lines:** Do the same for small **Bhaktapur** (`bhaktapur` from the example): count its cells with and without `all_touched=True`, and print the difference in percent, rounded to 2 decimal places.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
bkt_c = int(dem_da.rio.clip(bhaktapur.geometry, crs=bhaktapur.crs).notnull().sum())
bkt_t = int(dem_da.rio.clip(bhaktapur.geometry, crs=bhaktapur.crs, all_touched=True).notnull().sum())
print(bkt_c, bkt_t, round((bkt_t - bkt_c) / bkt_c * 100, 2))
```

145,368 and 146,512 cells, 0.79% more. The ring of border cells grows with the length of the border, and a small district has more border for its size.
</details>

**Q3 · three lines:** Estimate Bhaktapur's area from its cells (default clip) with `cell_km2`, and compare it with the area of the polygon in UTM 45N (B4). Print both in km², rounded to 1 decimal place.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
n_bkt = int(dem_da.rio.clip(bhaktapur.geometry, crs=bhaktapur.crs).notnull().sum())
print(round(n_bkt * cell_km2, 1))
print(round(bhaktapur.to_crs(32645).area.iloc[0] / 1000000, 1))
```

122.2 km² from the cells against 122.6 km² from the polygon, within 0.4%. Bhaktapur lies wholly inside the DEM, so no sliver is missing here; the small gap comes from the approximate cell area and the cells cut by the border.
</details>

## 7. Districts that lie only partly in the DEM

Clipping a polygon that reaches beyond the raster keeps only the part **inside**. Rioxarray does not warn you.

Nine districts touch the DEM (C2). Some lie almost wholly inside it; others only touch an edge. Any statistic from the clip describes **only the clipped part**. Sindhupalchok's mean elevation from this DEM is the mean of its south-western part, not of the whole district.

So before you report a number per district, measure how much of each district the raster covers. Below, a loop (A3) clips the DEM to each district, and the results go into a pandas table (A9). The share inside compares the area of the cells with the area of the polygon in UTM 45N, so it is approximate.

In [ ]:
touching = districts.cx[bounds.left:bounds.right, bounds.bottom:bounds.top]   # .cx from B5
areas_km2 = touching.to_crs(32645).area / 1000000                            # polygon areas, as in B4

records = []
for name, geometry, polygon_km2 in zip(touching["district"], touching.geometry, areas_km2):
    clip_da = dem_da.rio.clip([geometry], crs=districts.crs)
    n_cells = int(clip_da.notnull().sum())
    records.append({"district": name,
                    "inside_km2": round(n_cells * cell_km2, 1),
                    "district_km2": round(polygon_km2, 1),
                    "mean_m": round(float(clip_da.mean()), 1),
                    "max_m": float(clip_da.max())})

coverage = pd.DataFrame(records)
coverage["inside_pct"] = (coverage["inside_km2"] / coverage["district_km2"] * 100).round(1)
coverage.sort_values("inside_pct", ascending=False)

`zip` (B7) walks through three sequences at once: names, polygons and their areas in km². `dem_da.rio.clip([geometry], ...)` takes a **list** of shapes, so one polygon goes inside square brackets.

Bhaktapur (99.7%) and Kathmandu (99.1%) lie almost wholly inside. Nuwakot (46.9%), Kabhrepalanchok (42.4%), Lalitpur (40.7%) and Sindhupalchok (39.8%) are cut by the edges. Rasuwa, Makawanpur and Dhading only touch the DEM, with 2.2% or less.

The `mean_m` and `max_m` of the cut districts describe only their part inside. Sindhupalchok's maximum of 3,979 m is the highest cell of the **DEM** (C1), not of the district. Report such numbers with the share beside them, or leave them out.

### Practice

**Q1 · one line:** Show only the rows of `coverage` whose `inside_pct` is above 90 (B5 filtering).

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
coverage[coverage["inside_pct"] > 90]
```

Two rows: Bhaktapur and Kathmandu. Only for these two do the mean and maximum describe (nearly) the whole district.
</details>

**Q2 · two lines:** Which district holds the DEM's **lowest** cell? Clip the DEM to Nuwakot and print its lowest elevation.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
nuwakot = districts[districts["district"] == "Nuwakot"]
print(float(dem_da.rio.clip(nuwakot.geometry, crs=nuwakot.crs).min()))
```

525.0 m, the lowest value of the whole DEM (C1). It lies on the western edge, so the true lowest point of Nuwakot may well be outside the DEM.
</details>

**Q3 · three lines:** Add a column `note` to `coverage` that says `"whole"` where `inside_pct` is above 90 and `"part only"` elsewhere, using `np.where` (A8). Show the `district`, `inside_pct` and `note` columns.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
coverage["note"] = np.where(coverage["inside_pct"] > 90, "whole", "part only")
coverage[["district", "inside_pct", "note"]]
```

Bhaktapur and Kathmandu are `whole`; the other seven are `part only`. A column like this travels with the table, so nobody reads the clipped mean of Sindhupalchok as the mean of the whole district.
</details>

## 8. Saving a clip, and file sizes

`ktm_da.rio.to_raster(path)` writes a DataArray as a GeoTIFF, with its CRS and transform.

A DataArray opened with `masked=True` remembers the file's original type and nodata value. On writing, rioxarray turns `NaN` back into −32768 and stores `int16` again. So the clip costs 2 bytes per cell, not the 4 of `float32`.

By default, rioxarray writes **uncompressed** files. Add `compress="deflate"` for **lossless compression**: the file gets smaller and every value stays exactly the same. The DEM in `data/` is compressed this way (C1, step 7). Clips with many nodata cells shrink a lot, because long runs of the same value pack well.

As always, read the file back and check it.

In [ ]:
plain_path = OUT / "c3_kathmandu_dem.tif"
small_path = OUT / "c3_kathmandu_dem_deflate.tif"
ktm_da.rio.to_raster(plain_path)
ktm_da.rio.to_raster(small_path, compress="deflate")

for path in [DEM_PATH, plain_path, small_path]:
    with rasterio.open(path) as src:
        print(f"{path.name:32} {path.stat().st_size:>10,} bytes  {src.shape}  {src.dtypes[0]}  nodata={src.nodata}  compress={src.profile.get('compress')}")

In [ ]:
back_da = rxr.open_rasterio(small_path, masked=True).squeeze("band", drop=True)
print(back_da.shape, int(back_da.notnull().sum()), round(float(back_da.mean()), 1))
print(back_da.rio.crs, back_da.rio.bounds() == ktm_da.rio.bounds())

The whole DEM file takes about 2.77 million bytes. The uncompressed Kathmandu clip takes about 2.34 million bytes for a third of the cells, and the compressed clip only about 0.64 million (exact sizes vary a little between library versions). All three are `int16` with nodata −32768. `profile.get('compress')` gives `None` when there is no compression.

Read back, the compressed clip has the same shape, the same 488,217 valid cells, the same mean of 1,553.4 m, the same CRS and the same bounds (`True`). Compression lost nothing.

### Practice

**Q1 · one line:** Print the size of the compressed file as a percentage of the uncompressed one, rounded to 1 decimal place.

In [ ]:
# Q1: your answer

<details>
<summary><b>Show solution</b></summary>

```python
print(round(small_path.stat().st_size / plain_path.stat().st_size * 100, 1))
```

About 27.5%: deflate packs the clip into about a quarter of the space. (Your exact percentage may differ slightly with other library versions.)
</details>

**Q2 · two lines:** Save the Kathmandu Valley box `valley_da` from step 3 as `c3_valley_box.tif`, compressed, then print its size in bytes.

In [ ]:
# Q2: your answer

<details>
<summary><b>Show solution</b></summary>

```python
valley_path = OUT / "c3_valley_box.tif"
valley_da.rio.to_raster(valley_path, compress="deflate")
print(f"{valley_path.stat().st_size:,}")
```

About 1.6 million bytes. The box has no nodata cells, only varied elevations, so it compresses less well than the district clip with its large blank corners.
</details>

**Q3 · three lines:** Read `c3_valley_box.tif` back with rasterio and check it: print its shape, its CRS, and whether its highest value equals `valley_da.max()`.

In [ ]:
# Q3: your answer

<details>
<summary><b>Show solution</b></summary>

```python
with rasterio.open(OUT / "c3_valley_box.tif") as src:
    box_m = src.read(1)
    print(src.shape, src.crs)
print(box_m.max() == float(valley_da.max()))
```

(972, 1261), EPSG:4326, and `True`: 2,762 m in both. Rioxarray and rasterio read and write the same GeoTIFFs, so you can mix the two libraries freely.
</details>

## 9. When it breaks: polygons in the wrong CRS

In B4 you converted Kathmandu district to UTM 45N to measure it in metres. Here that UTM version goes into `clip`, without `crs=`. Run the cell.

In [ ]:
kathmandu_utm = kathmandu.to_crs(32645)       # metres, as in B4
dem_da.rio.clip(kathmandu_utm.geometry)

The last line of the error says:

`NoDataInBounds: No data found in bounds.`

- **NoDataInBounds** is rioxarray's error for "your shapes and the raster do not overlap".
- Without `crs=`, rioxarray assumes the shapes use the raster's CRS: degrees. The district's UTM eastings, between about 321,000 and 359,000 m, are read as longitudes, which lie nowhere near 85° E.

The same error appears when a polygon really lies outside the raster, such as a district in western Nepal.

The fix: bring the polygons to the raster's CRS first, and say which CRS they use:

```python
shapes = kathmandu_utm.to_crs(dem_da.rio.crs)
dem_da.rio.clip(shapes.geometry, crs=shapes.crs)
```

Checking `kathmandu.crs == dem_da.rio.crs` before clipping, as in step 4, catches this early.

**Try it:** fix the cell.

## Mixed review

These exercises combine everything in this lesson. Each one needs 5 to 10 lines. Plan in words first, then write the code.

### Review 1 · Save a window with rasterio

Write the window piece `valley_m` from step 1 to `c3_valley_window.tif` with rasterio, as in C1. Copy the DEM's `profile`, change `height`, `width` and `transform` with `.update(...)` (a dictionary method, A4), and write. Then read the file back and check that its values equal `valley_m` and its bounds are the window's bounds.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
with rasterio.open(DEM_PATH) as src:
    profile = src.profile.copy()
    win_bounds = src.window_bounds(win)

profile.update(height=valley_m.shape[0], width=valley_m.shape[1], transform=valley_transform)
window_path = OUT / "c3_valley_window.tif"
with rasterio.open(window_path, "w", **profile) as dst:
    dst.write(valley_m, 1)

with rasterio.open(window_path) as src:
    print(np.array_equal(src.read(1), valley_m))
    print(src.bounds)
    print(win_bounds)
```

`True`, and the two sets of bounds match: 85.1998…° to 85.5498…° E, 27.5501…° to 27.8201…° N. `**profile` unpacks the dictionary into keyword arguments, so the new file gets the DEM's dtype, nodata, CRS and compression, with its own size and transform. Forgetting to update the transform would put the piece at the DEM's top-left corner, 648 rows too far north.
</details>

### Review 2 · A map of Kathmandu district

Map the clipped Kathmandu DEM `ktm_da` with `land_cmap`, a colourbar with units and true proportions. Add the district outline and Kathmandu Durbar Square as a red triangle. Put the district's mean elevation in the title, and the Copernicus credit as a short line with `fig.text`.

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
mean_m = float(ktm_da.mean())

fig, ax = plt.subplots(figsize=(9, 6.5))
ktm_da.plot.imshow(ax=ax, cmap=land_cmap, cbar_kwargs={"label": "Elevation (m)"})
kathmandu.boundary.plot(ax=ax, color="black", linewidth=1)
ax.plot(85.307, 27.704, "r^", markersize=10)
ax.set_aspect(aspect)
ax.set_title(f"Kathmandu district: mean elevation {mean_m:,.0f} m (triangle: Durbar Square)")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
fig.text(0.01, 0.01, "Elevation: Copernicus DEM GLO-30, © DLR e.V. 2010-2014 and © Airbus Defence and Space GmbH 2014-2018, provided under COPERNICUS by the EU and ESA", fontsize=6)
plt.show()
```

The title reads a mean of 1,553 m. Durbar Square sits on the flat floor near the southern border of the district, and the high ground lies along the northern and western rims. `ax.set_xlabel` replaces the `x` label that `.plot.imshow` writes from the coordinate name.
</details>

### Review 3 · Kathmandu and Lalitpur compared

For Kathmandu and Lalitpur districts, clip the DEM and print the share of the district inside the DEM (from `coverage`), the mean elevation, and the percentage of the clipped cells above 2,000 m. Use a loop over the two names. Which number would you not report for Lalitpur without a warning?

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
for name in ["Kathmandu", "Lalitpur"]:
    district = districts[districts["district"] == name]
    clip_da = dem_da.rio.clip(district.geometry, crs=district.crs)
    n_inside = int(clip_da.notnull().sum())
    above_pct = int((clip_da > 2000).sum()) / n_inside * 100
    inside_pct = coverage.loc[coverage["district"] == name, "inside_pct"].iloc[0]
    print(f"{name:10} {inside_pct:5.1f}% inside  mean {float(clip_da.mean()):,.1f} m  {above_pct:.1f}% above 2,000 m")
```

Kathmandu: 99.1% inside, mean 1,553.4 m, 11.1% above 2,000 m. Lalitpur: 40.7% inside, mean 1,535.4 m, 5.5% above 2,000 m. Lalitpur's numbers describe only its northern part, inside the DEM. Both its mean and its share above 2,000 m need the warning "part only"; the rest of the district, outside the DEM, is not in them.
</details>

### Review 4 · The ground within 10 km of Chautara

Lesson B8 drew 10 km buffers around district headquarters in UTM 45N. Buffer Chautara by 10 km, bring the circle back to the DEM's CRS, and clip the DEM to it. Print the number of cells, their approximate area in km² (with `cell_km2`), the circle's area in km² from the UTM polygon, and the lowest, highest and mean elevation inside. Why is the area from the cells smaller?

In [ ]:
# Your answer

<details>
<summary><b>Show solution</b></summary>

```python
hq = gpd.read_file(DATA / "district_headquarters.geojson")
circle_utm = hq[hq["name"] == "Chautara"].to_crs(32645).buffer(10000)
circle = circle_utm.to_crs(dem_da.rio.crs)

near_da = dem_da.rio.clip(circle, crs=circle.crs)
n_near = int(near_da.notnull().sum())
print(n_near, round(n_near * cell_km2, 1), round(circle_utm.area.iloc[0] / 1000000, 1))
print(float(near_da.min()), float(near_da.max()), round(float(near_da.mean()), 1))
```

356,454 cells, about 299.7 km², against 313.7 km² for the circle. Chautara lies near the DEM's eastern edge (85.716° E), and 10 km east reaches beyond 85.8° E, so part of the circle has no cells. Inside, elevations run from 649 m to 2,975 m, with a mean of 1,340.2 m. The buffer was made in metres (B8) and converted to degrees **before** clipping, the fix from section 9.
</details>

## Recap

- A **window** (`Window(col_off, row_off, width, height)`, or `from_bounds(...)` rounded to whole cells) reads only part of a file; `window_transform` gives the piece its own transform. It equals a NumPy slice, rows first.
- `dem_da.rio.clip_box(minx, miny, maxx, maxy)` crops a DataArray to a box; `.sel` with slices works too, with y from top to bottom. Check shapes: edge cells cause off-by-one differences.
- `dem_da.rio.clip(shapes, crs=...)` keeps the cells whose **centre** lies in the polygons; `all_touched=True` keeps every touched cell; `drop=False` keeps the full grid. Shapes must be in the raster's CRS.
- Open with `masked=True` so outside cells become `NaN`. Without it they are nodata (−32768) and ruin every mean; check `min()` after a clip. `rasterio.mask.mask(..., crop=True)` is rasterio's tool for the same job.
- A clip of a district that reaches beyond the raster describes only the part inside: measure the share first. Save clips with `.rio.to_raster(path, compress="deflate")` and read them back.

### Words you learned

| Word | Meaning |
| --- | --- |
| subset | A part of a raster: some of its rows and columns, a box, or a polygon |
| window | A rectangle of cells inside a raster, given by its offsets, width and height |
| offset | Where a window starts: the column (`col_off`) and row (`row_off`) of its top-left cell |
| `window_transform` | The transform of a window: the raster's cell size with the window's own top-left corner |
| crop | Cut a raster down to a smaller rectangle |
| clip | Keep only the cells inside a shape; cells outside become `NaN` or nodata |
| mask | A stencil that hides cells: a True/False array, or a polygon used to clip |
| `all_touched` | Clip option: keep every cell the polygon touches, not only cells whose centre is inside |
| `drop` | rioxarray clip option: crop to the polygon's extent (`True`, default) or keep the full grid (`False`) |
| masked array | A NumPy array that carries a hidden list of cells to skip in calculations |
| `NoDataInBounds` | rioxarray's error when shapes and raster do not overlap, often a CRS mistake |
| lossless compression | Storing a file in fewer bytes while keeping every value exactly |

## Next

**C4 · Reproject, resample and mosaic:** move the DEM from degrees into UTM 45N with square 30 m cells, choose how values are resampled, make a coarser 90 m version, and stitch pieces back together.

<!-- pygile-nav -->
**PyGILE Course** · lesson 24 of 29 · [Course home](../README.md)  
← Previous: [C2 · Plotting rasters: an honest map of the Kathmandu Valley DEM](C02_Plotting_rasters.ipynb) · Next: [C4 · Reproject, resample and mosaic: the DEM on a grid in metres](C04_Reproject_resample_and_mosaic.ipynb) →